# Experiments 00 ? Standard project workflow

This notebook introduces the repeatable workflow used by project experiments. The goal is to keep data selection, downloading, cleaning, splitting, feature/target preparation, and model training in reusable workflow modules under `scripts/`. Experiment notebooks set the choices, call those modules, and explain the question and results.

This notebook moves the filtered HEST download and clean PNG/CSV export out of onboarding 01. It demonstrates the reusable downloader and the optional patch-level split. The default settings are a small preview; change the constants deliberately before a full download.

**Typical stages**
1. Choose a cohort and save its metadata.
2. Download and clean inputs using `scripts/hest_download_pipeline.py`.
3. Save a reproducible split manifest. For independent evaluation, group by patient or slide rather than randomly splitting patches.
4. Prepare features/targets and train using the appropriate helper; keep settings, manifests, and results together.

This helper-based workflow exports a clean `slide/patch/{png,csv,barcode.csv}` layout. It is an alternate data layout; onboarding 01?s HDF5/AnnData outputs remain the inputs for onboarding 02?03.


## 1. Install project dependencies and find the project modules
Script-based experiment notebooks install the project package list from `requirements.txt`. The project root is found from this notebook?s location so the code also works when launched from `notebooks/organized/`.

In [1]:
from pathlib import Path
import sys
import subprocess

here = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (here, *here.parents) if (p / 'requirements.txt').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Launch this notebook from the Hist2ST repository or a subfolder.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')])
SRC_DIR = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))
SCRIPT_DIR = PROJECT_ROOT / 'scripts'
sys.path.insert(0, str(SCRIPT_DIR))
print('Project root:', PROJECT_ROOT)
print('Reusable helpers:', SRC_DIR)
print('Workflow modules:', SCRIPT_DIR)


Project root: C:\Users\Siyu_\Documents\JHU_Files\courses\697_BDD\Hist2ST
Reusable helpers: C:\Users\Siyu_\Documents\JHU_Files\courses\697_BDD\Hist2ST\src
Workflow modules: C:\Users\Siyu_\Documents\JHU_Files\courses\697_BDD\Hist2ST\scripts


## 2. Set experiment paths and cohort choices
Keep data, manifests, and results under a chosen root. Change `DATA_DIR` for a local folder, Google Drive path, or Drive Desktop folder. `MAX_SAMPLES` is intentionally small for a first preview.

In [2]:
from huggingface_hub import login
from hest_download_pipeline import prepare_selection, download_clean_selection
from hest_preprocessing import random_split_data

DATA_DIR = (PROJECT_ROOT / 'data' / 'experiment_workflow').resolve()
CLEAN_DIR = DATA_DIR / 'clean_export'
SPLIT_DIR = DATA_DIR / 'splits' / 'preview_seed_42'
METADATA_FILE = 'HEST_v1_3_0.csv'
ORGANS = ['Breast']
SPECIES = ['Homo sapiens']
TECHNOLOGIES = ['Spatial Transcriptomics']
MAX_SAMPLES = 2  # Set to None to select every matching sample.
MAX_WORKERS = 2
SEED = 42
TRAIN_PERCENT = 80
RUN_DOWNLOAD = False  # Set True only after checking the selected IDs and storage space.
RUN_PATCH_SPLIT = False  # Set True after the clean export exists.
print('Data root:', DATA_DIR)


Data root: C:\Users\Siyu_\Documents\JHU_Files\courses\697_BDD\Hist2ST\data\experiment_workflow


d:\my-apps\anaconda3\envs\hist2st\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Sign in and select samples
The selection helper queries the HEST catalog with exact category filters. It returns editable IDs, so inspect them before allowing a large download.

In [3]:
login()  # Enter your Hugging Face token in the prompt.
data_ids = prepare_selection(
    organs=ORGANS,
    species=SPECIES,
    st_technologies=TECHNOLOGIES,
    metadata_filename=METADATA_FILE,
    max_samples=MAX_SAMPLES,
)
print('Selected IDs:', data_ids)
# Edit the list here if the experiment requires specific samples.
# data_ids = ['SPA100', 'SPA101']


Available organ: ['Bladder', 'Bone', 'Bowel', 'Brain', 'Breast', 'Cervix', 'Embryo', 'Eye', 'Heart', 'Kidney', 'Kidney/Brain', 'Liver', 'Lung', 'Lung/Brain', 'Lymph node', 'Lymphoid', 'Muscle', 'Other', 'Ovary', 'Pancreas', 'Placenta', 'Prostate', 'Skin', 'Spinal cord', 'Uterus', 'Whole organism']
Available species: ['Homo sapiens', 'Homo sapiens/Mus musculus', 'Mus musculus']
Available st_technology: ['Spatial Transcriptomics', 'Visium', 'Visium HD', "Visium HD 3'", 'Xenium', 'Xenium 5k']
Selected 2 samples
Selected IDs: ['SPA154', 'SPA153']


## 4. Download and clean the selected data
`download_clean_selection` reuses the HEST downloader, stages raw HDF5 files temporarily, and writes clean per-patch image/expression/barcode files. The download is disabled by default to prevent an accidental large transfer.

In [4]:
if RUN_DOWNLOAD:
    download_report = download_clean_selection(
        data_ids,
        output_dir=CLEAN_DIR,
        max_workers=MAX_WORKERS,
        metadata_filename=METADATA_FILE,
        staging_dir='/content' if Path('/content').is_dir() else None,
    )
    display(download_report)
    print('Clean data saved to:', CLEAN_DIR)
else:
    print('Download skipped. Review IDs and set RUN_DOWNLOAD=True to continue.')


Download skipped. Review IDs and set RUN_DOWNLOAD=True to continue.


## 5. Optional patch-level split
The helper writes ordered train/test manifests and a split configuration. This splits individual patches, so a slide or patient can appear on both sides; use it only for experiments where that is acceptable. The project?s patient-level pipeline uses explicit group maps and is the right choice for patient-independent evaluation.

In [5]:
if RUN_PATCH_SPLIT:
    if not CLEAN_DIR.is_dir():
        raise FileNotFoundError(f'Clean export not found: {CLEAN_DIR}. Run the download stage first.')
    train_paths, test_paths = random_split_data(
        data_folder=CLEAN_DIR,
        train_percentage=TRAIN_PERCENT,
        seed=SEED,
        output_dir=SPLIT_DIR,
    )
    print('Train patches:', len(train_paths))
    print('Test patches:', len(test_paths))
    print('Saved manifests and split settings under:', SPLIT_DIR)
else:
    print('Patch split skipped. Set RUN_PATCH_SPLIT=True after the clean export exists.')


Patch split skipped. Set RUN_PATCH_SPLIT=True after the clean export exists.


## 6. Which helper to use for later stages?

- **`hest_download_pipeline.py`:** catalog filtering, ID selection, HEST download, and clean export.
- **`hest_preprocessing.py`:** patch manifests/splits and ordered embedding or gene-tensor exports for clean patch folders. Its random patch split is not a patient-level split.
- **`hest_content_pipeline.py`:** the checkpointed cohort workflow, including audits, patient/slide-grouped splits, training-only gene selection, and target creation. Use its saved manifests and checkpoints to resume consistently.
- **`hest_transformer_training.py`:** the private Transformer architecture and training loop. Tensor loading and validation live in `src/load_tensors.py`; plotting is in `src/plot_figures.py`.

An experiment notebook should state its scientific question, cohort, split unit, settings/seed, outputs, and known limitations. It should call these shared functions for repeated mechanics, save its manifests/settings beside results, and plot the measurements needed to answer the question. Do not compare scores from different split units as if they measured the same generalization task.